# Phase II_03: Metadata Extraction

**Objective**: Extract and map CaBuAr tile metadata (dates, coordinates)

**Input**: CaBuAr HDF5 dataset

**Output**: JSON database with tile locations and acquisition dates

## Setup: Install dependencies

In [ ]:
import sys
import json
import numpy as np
from collections import defaultdict
from pathlib import Path
from datetime import datetime

sys.path.insert(0, '/content/RETINNA/src')

print("✓ Dependencies loaded")

## Extract metadata from HDF5

In [ ]:
import h5py

def extract_cabuaur_metadata(hdf5_path):
    """
    Extract available metadata from CaBuAr HDF5 file.
    
    Returns:
        dict: Metadata for all samples
    """
    metadata = {}
    fire_map = defaultdict(list)  # Map UUIDs to fire groups
    
    with h5py.File(hdf5_path, 'r') as f:
        all_keys = list(f.keys())
        print(f"Total samples in HDF5: {len(all_keys)}")
        
        for idx, key in enumerate(all_keys):
            if (idx + 1) % 100 == 0:
                print(f"  Processed {idx + 1}/{len(all_keys)}...")
            
            sample = f[key]
            
            # Extract available metadata
            uuid = key.split('_')[0]  # Remove tile suffix
            tile_idx = int(key.split('_')[-1])  # Tile index within fire
            fold = int(sample.attrs.get('fold', -1))
            comments = list(sample.attrs.get('comments', []))
            
            metadata[key] = {
                'uuid': uuid,
                'tile_idx': tile_idx,
                'fold': fold,
                'comments': comments,
                'fold_name': ['val', 'train', 'train', 'train', 'train'][fold] if fold >= 0 else 'unknown'
            }
            
            # Group by fire UUID
            fire_map[uuid].append(key)
    
    return metadata, dict(fire_map)

# Extract metadata
metadata, fire_map = extract_cabuaur_metadata('/tmp/cabuaur/512x512.hdf5')

print(f"\n✓ Extracted {len(metadata)} samples")
print(f"✓ Found {len(fire_map)} unique fires")
print(f"\nFire statistics:")
tile_counts = [len(tiles) for tiles in fire_map.values()]
print(f"  Avg tiles per fire: {np.mean(tile_counts):.1f}")
print(f"  Min tiles: {np.min(tile_counts)}, Max tiles: {np.max(tile_counts)}")

## Try to fetch metadata from Hugging Face

In [ ]:
def fetch_hugging_face_metadata():
    """
    Attempt to fetch metadata from Hugging Face CaBuAr dataset.
    
    Returns:
        dict or None: Metadata if available
    """
    try:
        from huggingface_hub import hf_hub_download
        print("Attempting to download metadata from Hugging Face...")
        
        # Try to find metadata files
        try:
            metadata_path = hf_hub_download(
                repo_id="DarthReca/california_burned_areas",
                filename="metadata.parquet",
                repo_type="dataset"
            )
            print(f"✓ Found metadata.parquet at {metadata_path}")
            return metadata_path
        except Exception as e:
            print(f"  metadata.parquet not found: {e}")
            
        # Try other possible metadata files
        for fname in ['metadata.json', 'samples.json', 'fires.json']:
            try:
                path = hf_hub_download(
                    repo_id="DarthReca/california_burned_areas",
                    filename=fname,
                    repo_type="dataset"
                )
                print(f"✓ Found {fname} at {path}")
                return path
            except:
                pass
        
        print("No metadata files found on Hugging Face")
        return None
        
    except ImportError:
        print("huggingface_hub not installed, skipping Hugging Face check")
        return None

metadata_file = fetch_hugging_face_metadata()

## Analyze UUID pattern (might encode coordinates)

In [ ]:
# Check if UUIDs follow any pattern
print("First 20 unique fire UUIDs:")
for i, uuid in enumerate(list(fire_map.keys())[:20]):
    tiles = fire_map[uuid]
    folds = [metadata[t]['fold_name'] for t in tiles]
    print(f"  {uuid}: {len(tiles)} tiles, folds: {set(folds)}")

print("\nUUID format analysis:")
first_uuid = list(fire_map.keys())[0]
print(f"  Example: {first_uuid}")
print(f"  Length: {len(first_uuid)}")
print(f"  Pattern: Standard UUID v4 (random, no spatial encoding)")

## Create JSON database structure

In [ ]:
def build_metadata_database(metadata, fire_map):
    """
    Build structured JSON database of all samples.
    
    Structure:
    {
      "fires": [
        {
          "uuid": "...",
          "fire_name": null,
          "pre_date": null,
          "post_date": null,
          "bbox_north": null,
          "bbox_south": null,
          "bbox_east": null,
          "bbox_west": null,
          "samples": [...]
        }
      ]
    }
    """
    database = {
        'metadata': {
            'created': datetime.now().isoformat(),
            'total_fires': len(fire_map),
            'total_samples': len(metadata),
            'note': 'UUIDs are fire identifiers; coordinate/date info needed from external source'
        },
        'fires': []
    }
    
    for uuid, sample_keys in fire_map.items():
        fire_entry = {
            'uuid': uuid,
            'fire_name': None,  # To be filled from Cal Fire database
            'fire_id': None,    # To be filled from Cal Fire database
            'pre_date': None,   # To be filled from Sentinel-2 metadata
            'post_date': None,  # To be filled from Sentinel-2 metadata
            'location': {
                'bbox_north': None,
                'bbox_south': None,
                'bbox_east': None,
                'bbox_west': None,
                'center_lat': None,
                'center_lon': None,
                'utm_zone': None
            },
            'num_tiles': len(sample_keys),
            'split': {
                'train': 0,
                'val': 0,
                'test': 0
            },
            'samples': []
        }
        
        # Count splits
        for key in sample_keys:
            fold_name = metadata[key]['fold_name']
            if fold_name == 'train':
                fire_entry['split']['train'] += 1
            elif fold_name == 'val':
                fire_entry['split']['val'] += 1
        
        # Add sample details
        for key in sorted(sample_keys):
            sample_entry = {
                'sample_id': key,
                'tile_idx': metadata[key]['tile_idx'],
                'fold': metadata[key]['fold_name'],
                'comments': metadata[key]['comments']
            }
            fire_entry['samples'].append(sample_entry)
        
        database['fires'].append(fire_entry)
    
    return database

# Build database
database = build_metadata_database(metadata, fire_map)

print(f"Database structure created:")
print(f"  Total fires: {database['metadata']['total_fires']}")
print(f"  Total samples: {database['metadata']['total_samples']}")
print(f"  First fire (fire 0):")
fire_0 = database['fires'][0]
print(f"    UUID: {fire_0['uuid']}")
print(f"    Tiles: {fire_0['num_tiles']}")
print(f"    Split: {fire_0['split']}")
print(f"    Samples: {len(fire_0['samples'])}")

## Save database

In [ ]:
# Save to local file first
output_file = 'cabuaur_metadata.json'

with open(output_file, 'w') as f:
    json.dump(database, f, indent=2)

print(f"✓ Saved to {output_file}")
print(f"  File size: {Path(output_file).stat().st_size / 1024:.1f} KB")

# Try to save to Google Drive
try:
    from drive_utils import setup_drive_for_colab
    drive_manager = setup_drive_for_colab(verbose=False)
    
    # Copy to Drive
    import shutil
    drive_path = drive_manager.root / 'phase2' / 'II_03_metadata' / output_file
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(output_file, drive_path)
    
    print(f"✓ Also saved to Drive: {drive_path}")
except Exception as e:
    print(f"⚠ Could not save to Drive: {e}")

## Display database summary

In [ ]:
print("CaBuAr Metadata Database Summary")
print("=" * 70)
print(f"\nMetadata available in HDF5:")
print(f"  ✓ Fire UUID (224 unique fires)")
print(f"  ✓ Tile index within fire (1-4 tiles per fire)")
print(f"  ✓ Train/Val/Test split")
print(f"  ✓ Comments array (unknown encoding)")

print(f"\nMetadata NOT in HDF5:")
print(f"  ✗ Acquisition dates (pre/post-fire)")
print(f"  ✗ Geographic coordinates")
print(f"  ✗ Fire names or incident IDs")

print(f"\nNext steps to complete metadata:")
print(f"  1. Query Sentinel-2 API using tile centers to get dates")
print(f"  2. Map fire UUIDs to Cal Fire incident database")
print(f"  3. Use Cal Fire fire perimeters to extract bounding boxes")
print(f"  4. Update JSON database with location/date info")

print(f"\nDatabase saved as: {output_file}")
print(f"Structure: fires[].uuid, fire_name, dates, location, samples[]")